# Spark sources and sinks
## Spark data sources
### External data source
- Lets assume that we have a batch processing requirement 
- Use a data integration tool to bring data to the distributed storage and then we start processing it.
- The reason for preferring this two step approach : 
    - Modularity
        - Bringing data correctly and effitiently to your lake is a complex goal in itself. The idea here is to decouple the ingestion from the processing to improve the managability.
    - Load Balance
        - The source systems would have been designed for a specific purpose and the capacity of the source systems would have been planned accordingly
    - Security
        - Now if you want to connect your spark work load to these systems then you must have to re-plan your source system's capacity and those security aspects of those systems.
    - Flexibility
        - We wnat to use the right tool for the right purpose.
        - Spark was designed for data processing its not suited for data ingestion.
        - Hence even though spark provide a way to connect to an external source usually people avoid to do so.
### Internal data source
- Internal data source could be HDFS or some cloud storage
- The mechanism of reading data is the same in case of HDFS and cloud storage. The only difference is in the file format.

### NOTE : 
- Working with the data source is all about reading the data. 
- Working with the sink is all about writing the data.


## Spark DataFrame reader API
### How to use dataFrame reader for csv, json and parquet data sources
- When we import data from csv or json file format and use inferschema to true in case of csv then the datatypes of the columns are mostly correct except for the date type columns. The datatype of the date type columns for some reason is always set to string 
- In case of importing data from a parquet file we don't have to worry about the dataType of the columns because it contains the shema information already included in the data file. So in this case I don't have to specify the schema explicitly. However the data-file must contain the correct schema.
- Because of this reason it is recommended to use parquet file format for as long as possible.
### Explicitly set the schema for you dataFrames
- DataFrame schema is all about setting the column name and its appropriate dataTypes, However you should also know the spark supported dataTypes.
- Spark DataTypes : Apache spark comes with its own dataTypes when it comes to defining the spark dataFrame schema.
    - IntegerType
    - LongType
    - FloatType
    - DoubleType
    - StringType
    - DateType
    - TimestampType
    - ArrayType
    - MapType
- Why does spark maintains its own datatypes instead of using the language specific types? for example python has its own datatypes why don't we use python dataTypes to define the dataFrame schema?
    - Spark is like a compiler it compiles the high level api code into lower level RDD operations.
    - During this compilation process it generates different execution plans and also perform a bunch of optimizations. All this is not possible without maintaining its own type information.
- Spark allows you to define the spark dataFrame schema in two ways 
    - Programmatically
        - schema.py
        ```python
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType

        """
        This class defines the spark dataFrame schema using Programmatical method
        """
        class FlightSchemaMixin:
            def return_flight_df_schema(self):
                flight_schema = StructType([
                    StructField("FL_DATE", DateType(), True),
                    StructField("OP_CARRIER", StringType(), True),
                    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
                    StructField("ORIGIN", StringType(), True),
                    StructField("ORIGIN_CITY_NAME", StringType(), True),
                    StructField("DEST", StringType(), True),
                    StructField("DEST_CITY_NAME", StringType(), True),
                    StructField("CRS_DEP_TIME", IntegerType(), True),
                    StructField("DEP_TIME", IntegerType(), True),
                    StructField("WHEELS_ON", IntegerType(), True),
                    StructField("TAXI_IN", IntegerType(), True),
                    StructField("CRS_ARR_TIME", IntegerType(), True),
                    StructField("ARR_TIME", IntegerType(), True),
                    StructField("CANCELLED", IntegerType(), True),
                    StructField("DISTANCE", IntegerType(), True)
                ])
                return flight_schema
        ```
        - ingest.py
        ```python
        # --- Add project root to sys.path ---
        import os 
        import sys
        CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
        PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
        if PROJECT_ROOT not in sys.path:
            sys.path.insert(0, PROJECT_ROOT)

        from .logger import Log4j
        from .app_monitor import GetDataFrameMemory

        from spark_dataFrame_schema.spark_dataframe_schema import FlightSchemaMixin

        """
        This class ingest data from csv, json and parquet file format
        """
        class IngestData():
            def __init__(self,spark):
                self.spark_object = spark
                self.logger = Log4j(spark)
                self.metrics = GetDataFrameMemory(spark)
                self.df_schema = FlightSchemaMixin()

            def import_data_csv(self,file_dir):
                try:
                    spark_df = (
                            self.spark_object
                            .read
                            .format("csv")
                            .option("header","true")
                            # .option("inferschema","true")
                            .schema(self.df_schema.return_flight_df_schema())
                            # Set the mode for error if the schema don't match
                            .option("mode","FAILFAST")
                            # Set the date string format
                            .option("dateFormat","M/d/y")
                            .load(file_dir)
                    )
                    self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
                    return spark_df
                except Exception as e:
                    self.logger.error(str(e))

            # utility methods
            def log_df_metrics(self,spark_df,file_dir):
                self.logger.info(f"import_data_csv :: spark_df created successfully from {file_dir} dataset file")
                self.logger.info(f"import_data_csv :: The memory taken by the spark dataFrame is = {self.metrics.get_mem_usage(spark_df).get("mem")} MB")
                schema_str = spark_df._jdf.schema().treeString()
                self.logger.debug(f"Spark DataFrame Schema (expanded): {schema_str}")
        ```
        - main.py
        ```python
        from pyspark.sql import SparkSession
        # import related to logging
        from lib.logger import Log4j, LogSparkDataframe
        # import related to custom spark configurations
        from lib.utils import get_spark_app_config
        # logging related imports 
        import os

        # Imports related to ingest data
        from lib.ingest_data import IngestData
        # Transform data
        from transformations.dataframe_transformations import DataFrameTransformations

        if __name__ == "__main__":
            # logging related logic
            # Get the current project's directory
            project_dir = os.path.dirname(os.path.abspath(__file__))
            # Get the Log4j.properties file directory
            log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
            # Save the directory where the generated log files must reside
            log_dir = os.path.join(project_dir, "log4j_properties", "logs")
            # Create the directory where the log files must be kept if not present
            os.makedirs(log_dir, exist_ok=True)

            conf = get_spark_app_config()
            spark = (
                SparkSession
                .builder
                .config(conf=conf)
                .config("spark.driver.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.executor.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .getOrCreate()
            )

            # initialize logger class 
            logger = Log4j(spark)

            # initialize the spark dataframe logger 
            sp_df_logger = LogSparkDataframe(spark)

            # logging some debug related stuff 
            logger.debug(f"log4j.properties file dir = {log4j_config_path}")
            logger.debug(f"log files dir = {log_dir}")
            logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
            
            logger.info("Reading the data from the directory")
            dataset_dir = os.path.join(project_dir,"dataset")
            # file_name = "sf-fire-calls.csv" nor mally we provide the file name by hard coding it in the app 
            # But here the dataset file name is supplied via spark.conf file
            file_name = conf.get("file_name_csv")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_csv dir = {file_dir}")
            
            # The function must taken in file_dir csv file and then returns a spark dataFrame
            # import data from a csv file
            ingest_data = IngestData(spark)
            spark_df = ingest_data.import_data_csv(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df")

            # import data from a json file
            file_name = conf.get("file_name_json")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_json = ingest_data.import_data_json(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_json")

            # import data from a parquet file
            file_name = conf.get("file_name_parquet")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_parquet = ingest_data.import_data_parquet(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_parquet")

            # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
            # input("Please enter")
            spark.stop()
        ```
    - Using DDL String
        - schema.py
        ```python
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType


        class FlightSchemaMixin:

            """
            This method defines the spark dataFrame schema using DDL method
            """
            def return_flight_schema_ddl(self):
                flight_schema_ddl = """
                    FL_DATE DATE,
                    OP_CARRIER STRING,
                    OP_CARRIER_FL_NUM INT,
                    ORIGIN STRING,
                    ORIGIN_CITY_NAME STRING,
                    DEST STRING,
                    DEST_CITY_NAME STRING,
                    CRS_DEP_TIME INT,
                    DEP_TIME INT,
                    WHEELS_ON INT,
                    TAXI_IN INT,
                    CRS_ARR_TIME INT,
                    ARR_TIME INT,
                    CANCELLED INT,
                    DISTANCE INT
                """
                return flight_schema_ddl
        ```
        - ingest_data.py
        ```python
        # --- Add project root to sys.path ---
        import os 
        import sys
        CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
        PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
        if PROJECT_ROOT not in sys.path:
            sys.path.insert(0, PROJECT_ROOT)

        from .logger import Log4j
        from .app_monitor import GetDataFrameMemory

        from spark_dataFrame_schema.spark_dataframe_schema import FlightSchemaMixin

        """
        This class ingest data from csv, json and parquet file format
        """
        class IngestData():
            def __init__(self,spark):
                self.spark_object = spark
                self.logger = Log4j(spark)
                self.metrics = GetDataFrameMemory(spark)
                self.df_schema = FlightSchemaMixin()

            def import_data_json(self,file_dir):
                try:
                    spark_df = (
                        self.spark_object
                        .read
                        .format("json")
                        .schema(self.df_schema.return_flight_schema_ddl())
                        .option("dateFormat","M/d/y")
                        .load(file_dir)
                    )
                    self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
                    return spark_df
                except Exception as e:
                    self.logger.error(str(e))

            def import_data_parquet(self,file_dir):
                try:
                    spark_df = (
                        self.spark_object
                        .read
                        .format("parquet")
                        .load(file_dir)
                    )
                    self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
                    return spark_df
                except Exception as e:
                    self.logger.error(str(e))

            # utility methods
            def log_df_metrics(self,spark_df,file_dir):
                self.logger.info(f"import_data_csv :: spark_df created successfully from {file_dir} dataset file")
                self.logger.info(f"import_data_csv :: The memory taken by the spark dataFrame is = {self.metrics.get_mem_usage(spark_df).get("mem")} MB")
                schema_str = spark_df._jdf.schema().treeString()
                self.logger.debug(f"Spark DataFrame Schema (expanded): {schema_str}")
         ```
         - main.py
         ```python
         from pyspark.sql import SparkSession
        # import related to logging
        from lib.logger import Log4j, LogSparkDataframe
        # import related to custom spark configurations
        from lib.utils import get_spark_app_config
        # imports related to exporting dataframe
        from lib.write_df import ExportSparkDataFrame
        # logging related imports 
        import os

        # Imports related to ingest data
        from lib.ingest_data import IngestData
        # Transform data
        from transformations.dataframe_transformations import DataFrameTransformations

        if __name__ == "__main__":
            # logging related logic
            # Get the current project's directory
            project_dir = os.path.dirname(os.path.abspath(__file__))
            # Get the Log4j.properties file directory
            log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
            # Save the directory where the generated log files must reside
            log_dir = os.path.join(project_dir, "log4j_properties", "logs")
            # Create the directory where the log files must be kept if not present
            os.makedirs(log_dir, exist_ok=True)

            conf = get_spark_app_config()
            spark = (
                SparkSession
                .builder
                .config(conf=conf)
                .config("spark.driver.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.executor.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .getOrCreate()
            )

            # initialize logger class 
            logger = Log4j(spark)

            # initialize the spark dataframe logger 
            sp_df_logger = LogSparkDataframe(spark)

            # logging some debug related stuff 
            logger.debug(f"log4j.properties file dir = {log4j_config_path}")
            logger.debug(f"log files dir = {log_dir}")
            logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
            
            logger.info("Reading the data from the directory")
            dataset_dir = os.path.join(project_dir,"dataset")

            # INGETING DATA FROM VARIOUS FILE FORMATS STARTS
            # file_name = "sf-fire-calls.csv" nor mally we provide the file name by hard coding it in the app 
            # But here the dataset file name is supplied via spark.conf file
            file_name = conf.get("file_name_csv")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_csv dir = {file_dir}")
            
            # The function must taken in file_dir csv file and then returns a spark dataFrame
            # import data from a csv file
            ingest_data = IngestData(spark)
            spark_df = ingest_data.import_data_csv(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df")

            # import data from a json file
            file_name = conf.get("file_name_json")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_json = ingest_data.import_data_json(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_json")

            # import data from a parquet file
            file_name = conf.get("file_name_parquet")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            spark_df_parquet = ingest_data.import_data_parquet(file_dir=file_dir)

            # log spark dataframe
            sp_df_logger.log_df(spark_df=spark_df,spark_df_name="spark_df_parquet")
            # INGETING DATA FROM VARIOUS FILE FORMATS ENDS

            # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
            # input("Please enter")
            spark.stop()
         ```
- Spark dataframe schema is of StructType which is made up of a list of StructField
    - The StructField takes in two argument one is the column name and the second one is the dataType.
    - Spark must throw an error if the dataFrame schema doesn't match with the incoming data from the csv or json file however we will have to setup the mode for getting the error or else spark will sielently fail to parse the column whose datatype don't match with the defined spark schema.
### Export spark dataFrame in paraquet format
- There a some key things to remeber before we move forward 
    - When exporting dataFrames you have to make sure that the dataFrame is partition in such a way that the partition file size in your file system ranges from 500MB to 1GB not too small and not too big
#### Errors I faced 
```bash
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/aditya/github/Deep-learning-prerequisite/pyspark/Spark_sources_and_sinks/SparkSchemaDemo/main_sch_app.py", line 37, in <module>
    .getOrCreate()
     ~~~~~~~~~~~^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/sql/session.py", line 556, in getOrCreate
    sc = SparkContext.getOrCreate(sparkConf)
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/core/context.py", line 523, in getOrCreate
    SparkContext(conf=conf or SparkConf())
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/core/context.py", line 205, in __init__
    SparkContext._ensure_initialized(self, gateway=gateway, conf=conf)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/core/context.py", line 444, in _ensure_initialized
    SparkContext._gateway = gateway or launch_gateway(conf)
                                       ~~~~~~~~~~~~~~^^^^^^
  File "/home/aditya/miniconda3/envs/pyspark/lib/python3.13/site-packages/pyspark/java_gateway.py", line 111, in launch_gateway
    raise PySparkRuntimeError(
    ...<2 lines>...
    )
pyspark.errors.exceptions.base.PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.
``` 
- This error occurs due to mismatch of scala version. Remember the same pyspark version can be compiled wih different scala version.
- In my case I added this config 
```python
.config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:4.0.1")
```
- After getting this error I confirmed which version of scala is being used under the hood of my pyspark installation by typing 
```bash
conda activate pyspark --> activate your env
pyspark --version
```
You should get output like this 
```bash
WARNING: Using incubator modules: jdk.incubator.vector
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/02 17:44:32 WARN Utils: Your hostname, aditya-IdeaPad-5-15ITL05, resolves to a loopback address: 127.0.1.1; using 192.168.1.19 instead (on interface wlp0s20f3)
25/11/02 17:44:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.1
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.15
Branch HEAD
Compiled by user runner on 2025-09-02T03:10:51Z
Revision 29434ea766b0fc3c3bf6eaadb43a8f931133649e
Url https://github.com/apache/spark
Type --help for more information.

```
- As you can see there is a clear mismatch in the scala version if you compare these two ```Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.15``` vs ```.config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:4.0.1")```
#### Solution
- Update the import when declaring dependecis during Spark session building process ```.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.0.1")```